In [6]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/interim/subreddits/WomensHealth/submissions")

# If memory is okay
df = pd.read_json(file_path, lines=True)

# If you want a chunked, lower-memory load:
# chunks = pd.read_json(file_path, lines=True, chunksize=100000)
# df = pd.concat(chunks, ignore_index=True)

In [7]:
# create new dataframe that keeps only: created_utc, title, selftext, upvote_ratio, ups, num_comments, media, and subreddit
df_subset = df[['id', 'title', 'selftext', 'upvote_ratio', 'ups', 'num_comments', 'subreddit']]
df_subset.isnull().sum()

id                  0
title               0
selftext            0
upvote_ratio    17973
ups             58287
num_comments        0
subreddit           0
dtype: int64

Add df_subset to postgres database as submissions table

In [ ]:
# pip install psycopg2

     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     - -------------------------------------- 0.1/2.8 MB 3.2 MB/s eta 0:00:01
     ------- -------------------------------- 0.5/2.8 MB 6.8 MB/s eta 0:00:01
     --------------- ------------------------ 1.0/2.8 MB 9.4 MB/s eta 0:00:01
     --------------- ------------------------ 1.0/2.8 MB 9.4 MB/s eta 0:00:01
     ------------------------------ --------- 2.1/2.8 MB 10.2 MB/s eta 0:00:01
     ---------------------------------------  2.8/2.8 MB 10.3 MB/s eta 0:00:01
     ---------------------------------------- 2.8/2.8 MB 8.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: C:\Users\addie\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv 
# loading variables from .env file
load_dotenv() 

True

In [72]:
import psycopg2

# Connect to the database
conn = psycopg2.connect(
    dbname=os.getenv("DBNAME"),
    user=os.getenv("DBUSER"),
    password=os.getenv("DBPASSWORD"),
    port=os.getenv("DBPORT"),
    host=os.getenv("DBHOST")
)
conn_string=os.getenv("CONNSTRING")

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

In [76]:
df_subset.to_sql("submissions", engine, if_exists="append", index=False, method="multi", chunksize=1000)

123997